# Single DataStream Analysis

This notebook walks through the complete workflow for analysing a **single simulation run** with `quends`:

1. Load a CSV file into a `DataStream`
2. Explore the data (shape, variables, head)
3. Check for stationarity
4. Visualise the raw time series
5. Trim the transient using the explicit **strategy-operation pattern**
6. Compute effective sample size (ESS) and statistics
7. Export results
8. Compare trim strategies
9. Inspect cumulative statistics for convergence

**Data assumed**: `gx/tprim_2_5_a.out.csv` relative to the notebook's working directory.

In [ ]:
import quends as qnds

print(f"quends version: {qnds.__version__}")

## 1. Load Data

`qnds.from_csv()` reads a CSV file and returns a `DataStream`.  The CSV is expected to have a `time` column plus one or more quantity columns (e.g. `HeatFlux_st`, `ParticleFlux_st`).

In [ ]:
ds = qnds.from_csv("gx/tprim_2_5_a.out.csv")

print(type(ds))

## 2. Explore the DataStream

A `DataStream` exposes several convenience methods for quick inspection.

In [ ]:
# Available quantity columns (excludes the time index)
print("Variables:", ds.variables())

In [ ]:
# First few rows of the underlying DataFrame
ds.head()

In [ ]:
# Total number of time steps
print(f"Number of time steps: {len(ds)}")

## 3. Stationarity Check

`DataStream.is_stationary(col)` runs an augmented Dickey-Fuller (ADF) test on `col` and returns `True` if the series is judged stationary at the standard significance level.  Running this on the **full** (untrimmed) stream is useful for confirming that a transient is present.

In [ ]:
col = "HeatFlux_st"

stationary_raw = ds.is_stationary(col)
print(f"Is the full (raw) series stationary? {stationary_raw}")
# Typically False — the warm-up phase drives non-stationarity

## 4. Visualise the Raw Time Series

`qnds.Plotter()` provides plotting helpers that return Matplotlib figures.  `trace_plot` shows the raw time series for one or more columns.

In [ ]:
%matplotlib inline

plotter = qnds.Plotter()
plotter.trace_plot(ds, ["HeatFlux_st"])

## 5. Trim the Transient — QuantileTrimStrategy

All trimming uses the **explicit strategy-operation pattern**:

```
strategy = <StrategyClass>(window_size=..., ...)
op       = TrimDataStreamOperation(strategy=strategy)
trimmed  = op(ds, column_name="HeatFlux_st")
```

`QuantileTrimStrategy` (factory key `"std"`) identifies the start of steady state by looking for the first window where the signal lies within `robust` bounds (MAD-based when `robust=True`, std-based otherwise).

In [ ]:
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation

# Step 1: configure the strategy
strategy = QuantileTrimStrategy(window_size=20, robust=True)

# Step 2: wrap it in the operation
op = TrimDataStreamOperation(strategy=strategy)

# Step 3: apply to the DataStream
trimmed = op(ds, column_name="HeatFlux_st")

print(f"Raw length     : {len(ds)} steps")
print(f"Trimmed length : {len(trimmed)} steps")
print(f"Steady state starts at t = {trimmed.data['time'].iloc[0]:.3f}")

### Visualise the Trim Result

`steady_state_automatic_plot` overlays the detected steady-state window on the trace, making it easy to verify that the trim is sensible.

In [ ]:
plotter.steady_state_automatic_plot(ds, ["HeatFlux_st"])

## 6. Effective Sample Size and Statistics

Plasma heat-flux time series are strongly autocorrelated, so the number of statistically independent samples is much smaller than the raw time-step count.  `effective_sample_size()` computes the ESS using integrated autocorrelation time.

In [ ]:
ess = trimmed.effective_sample_size()
print(f"Effective sample size (ESS): {ess}")

In [ ]:
# Compute sliding-window statistics for each variable in the trimmed stream
stats = trimmed.compute_statistics(method="sliding")
print(stats)

In [ ]:
# Mean and 95% confidence interval for HeatFlux_st
mean_val = trimmed.mean()
ci       = trimmed.confidence_interval()
print(f"Mean             : {mean_val}")
print(f"Confidence interval (95%): {ci}")

## 7. Export Results

`qnds.Exporter()` provides formatted display helpers.  `display_dataframe` renders statistics as a styled pandas DataFrame; `display_json` emits structured JSON for programmatic downstream use.

In [ ]:
exporter = qnds.Exporter()

# Display the HeatFlux_st statistics as a formatted table
exporter.display_dataframe(stats["HeatFlux_st"])

In [ ]:
# Or as JSON (useful for saving to disk / passing to downstream tools)
exporter.display_json(stats["HeatFlux_st"])

## 8. Comparing Trim Strategies

Because the strategy and operation are separate objects, switching strategies requires only changing the strategy class.  The operation call site is identical.

Here we try `IQRTrimStrategy` (factory key `"iqr"`) as an alternative.

In [ ]:
from quends.base.trim import IQRTrimStrategy

# IQR strategy: threshold controls how many IQR-widths from the median are tolerated
strategy2 = IQRTrimStrategy(window_size=20, threshold=0.05)
op2       = TrimDataStreamOperation(strategy=strategy2)
trimmed2  = op2(ds, column_name="HeatFlux_st")

print(f"QuantileTrim start index : {len(ds) - len(trimmed)}  (t = {trimmed.data['time'].iloc[0]:.3f})")
print(f"IQRTrim start index      : {len(ds) - len(trimmed2)} (t = {trimmed2.data['time'].iloc[0]:.3f})")

In [ ]:
# Compute statistics for the IQR-trimmed stream and compare means
stats2 = trimmed2.compute_statistics(method="sliding")

mean1 = trimmed.mean()
mean2 = trimmed2.mean()
print(f"Mean (QuantileTrim): {mean1}")
print(f"Mean (IQRTrim)     : {mean2}")

## 9. Cumulative Statistics — Convergence Diagnostic

`cumulative_statistics()` returns the running mean and running standard error as a function of the number of time steps included.  A flat plateau indicates that the statistics have converged and the trimmed window is long enough.

In [ ]:
cumstats = trimmed.cumulative_statistics()
print("Cumulative statistics (first 10 rows):")
print(cumstats.head(10))

In [ ]:
# Visualise the ACF to confirm autocorrelation structure
plotter.plot_acf(trimmed)

## Summary

| Step | Method |
|------|--------|
| Load | `qnds.from_csv(path)` |
| Explore | `.variables()`, `.head()`, `len(ds)` |
| Stationarity | `.is_stationary(col)` |
| Visualise | `Plotter().trace_plot(ds, cols)` |
| Trim | `QuantileTrimStrategy` + `TrimDataStreamOperation` |
| ESS | `.effective_sample_size()` |
| Statistics | `.compute_statistics()`, `.mean()`, `.confidence_interval()` |
| Export | `Exporter().display_dataframe(stats_df)` |
| Convergence | `.cumulative_statistics()`, `Plotter().plot_acf(ds)` |

**Next**: `03_technique0_ensemble.ipynb` — ensemble analysis with Technique 0.